# Machine Learning Workflow — Customer Loyalty History

**Objective:** Predict the customer's `Loyalty Card` category using customer profile and enrollment information.

- **Dataset:** Customer Loyalty History.csv
- **Target:** `Loyalty Card`
- **Model:** Random Forest Classifier
- **Split:** 80% training / 20% testing
- **Evaluation:** Accuracy, Precision, Recall, F1-score, and Support

**Important:** In Google Colab, run the first code cell and upload `Customer Loyalty History.csv` when prompted. The remaining cells are designed to run in order.

## Task 1 — Dataset Selection and Exploration
Load the CSV file and inspect the rows, columns, data types, missing values, and target distribution.

In [1]:
import os
import pandas as pd
import numpy as np

DATA_PATH = "/content/Customer Loyalty History.csv"

# If the CSV is not already available, prompt you to upload it in Google Colab.
if not os.path.exists(DATA_PATH):
    try:
        from google.colab import files
        print("Upload the file: Customer Loyalty History.csv")
        uploaded = files.upload()
        csv_files = [name for name in uploaded.keys() if name.lower().endswith(".csv")]
        if not csv_files:
            raise FileNotFoundError("No CSV file was uploaded. Please upload Customer Loyalty History.csv.")
        DATA_PATH = csv_files[0]
    except ImportError:
        DATA_PATH = "Customer Loyalty History.csv"
        if not os.path.exists(DATA_PATH):
            raise FileNotFoundError(
                "CSV file not found. Put 'Customer Loyalty History.csv' in the same folder as this notebook."
            )

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())
print("\nColumn data types:")
display(df.dtypes.to_frame(name="Data Type"))
print("\nMissing values per column:")
display(df.isnull().sum().sort_values(ascending=False).to_frame(name="Missing Values"))
print("\nTarget distribution:")
display(df["Loyalty Card"].value_counts().to_frame(name="Count"))

Dataset shape: (16737, 16)


,Loyalty Number,Country,Province,City,Postal Code,Gender,Education,Salary,Marital Status,Loyalty Card,CLV,Enrollment Type,Enrollment Year,Enrollment Month,Cancellation Year,Cancellation Month
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,83236.0,Married,Star,3839.14,Standard,2016,2,NaN,NaN
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,NaN,Divorced,Star,3839.61,Standard,2016,3,NaN,NaN
2,429460,Canada,British Columbia,Vancouver,V6E 3D9,Male,College,NaN,Single,Star,3839.75,Standard,2014,7,2018.0,1.0
3,608370,Canada,Ontario,Toronto,P1W 1K4,Male,College,NaN,Single,Star,3839.75,Standard,2013,2,NaN,NaN
4,530508,Canada,Quebec,Hull,J8Y 3Z5,Male,Bachelor,103495.0,Married,Star,3842.79,Standard,2014,10,NaN,NaN



Column data types:


,Data Type
Loyalty Number,int64
Country,object
Province,object
City,object
Postal Code,object
Gender,object
Education,object
Salary,float64
Marital Status,object
Loyalty Card,object



Missing values per column:


,Missing Values
Cancellation Month,14670
Cancellation Year,14670
Salary,4238
Loyalty Number,0
City,0
Postal Code,0
Province,0
Country,0
Education,0
Gender,0



Target distribution:


,Count
Loyalty Card,
Star,7637
Nova,5671
Aurora,3429


## Task 2 — Data Preprocessing

- `Loyalty Card` is the target.
- Remove `Loyalty Number` because it is an identifier.
- Remove cancellation year/month for this chosen task.
- Fill missing numeric values using the median.
- Fill missing categorical values using the most frequent category.
- One-hot encode categorical features.
- Keep preprocessing inside a pipeline to prevent data leakage.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

TARGET = "Loyalty Card"
DROP_COLUMNS = ["Loyalty Number", "Cancellation Year", "Cancellation Month"]

# Validate that the expected target exists before proceeding.
if TARGET not in df.columns:
    raise ValueError(f"Target column '{TARGET}' was not found. Available columns: {df.columns.tolist()}")

# Remove rows without a target label, if any.
df = df.dropna(subset=[TARGET]).copy()

X = df.drop(columns=[TARGET] + [col for col in DROP_COLUMNS if col in df.columns])
y = df[TARGET]

categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_features = X.select_dtypes(include=["number"]).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("Rows available for modeling:", len(df))
print("Number of input features:", X.shape[1])
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Missing target values after cleaning:", y.isna().sum())

Rows available for modeling: 16737
Number of input features: 12
Numeric features: ['Salary', 'CLV', 'Enrollment Year', 'Enrollment Month']
Categorical features: ['Country', 'Province', 'City', 'Postal Code', 'Gender', 'Education', 'Marital Status', 'Enrollment Type']
Missing target values after cleaning: 0


## Task 3 — Data Splitting
Split the data into 80% training and 20% testing. Stratification keeps class proportions similar across the two sets.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)
print("\nTraining class distribution (%):")
display((y_train.value_counts(normalize=True) * 100).round(2).to_frame(name="Percent"))
print("\nTesting class distribution (%):")
display((y_test.value_counts(normalize=True) * 100).round(2).to_frame(name="Percent"))

Training set: (13389, 12)
Testing set: (3348, 12)

Training class distribution (%):


,Percent
Loyalty Card,
Star,45.63
Nova,33.89
Aurora,20.49



Testing class distribution (%):


,Percent
Loyalty Card,
Star,45.64
Nova,33.87
Aurora,20.49


## Task 4 — Model Selection and Training
Train a Random Forest Classifier for multiclass classification. Class weighting is enabled to help account for differences in class frequency.

In [4]:
from sklearn.ensemble import RandomForestClassifier

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ))
])

model.fit(X_train, y_train)
print("Model training completed successfully.")

Model training completed successfully.


## Task 5 — Model Evaluation
Use the held-out test set to calculate Accuracy and the classification report.

In [5]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy Score: {accuracy:.4f} ({accuracy * 100:.2f}%)")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Accuracy Score: 0.4639 (46.39%)

Classification Report:
              precision    recall  f1-score   support

      Aurora       0.31      0.27      0.29       686
        Nova       0.40      0.38      0.39      1134
        Star       0.56      0.61      0.58      1528

    accuracy                           0.46      3348
   macro avg       0.42      0.42      0.42      3348
weighted avg       0.45      0.46      0.46      3348



## Conclusion

The Random Forest Classifier was trained to predict the customer's loyalty-card category. Use the test-set accuracy and the class-level precision, recall, and F1-scores printed above to assess performance. Precision indicates how often predictions for a class are correct; recall indicates how many actual examples of a class were found; and F1-score balances precision and recall. Support is the number of test examples in each class.

The model's performance should be judged using both overall accuracy and the per-class scores. If results are weak, potential next steps include comparing other classifiers, tuning hyperparameters, and reviewing feature relevance. Report the metrics generated by running this notebook rather than estimating or manually changing them.